In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#**Loading Cleaned Data**

Import the dataset generated from the EDA phase.


In [3]:
try:
    df = pd.read_csv('concrete_cleaned.csv')
    print("Successfully loaded 'concrete_cleaned.csv'")
except FileNotFoundError:
    print("Error: Please run File 1 first to generate the cleaned data.")

Successfully loaded 'concrete_cleaned.csv'


#**DOMAIN-SPECIFIC FEATURE ENGINEERING**

Create interaction terms based on Concrete Science.

**Interaction 1: Water-to-Cement Ratio (The most critical factor in concrete strength)**

In [4]:
df['Water_Cement_Ratio'] = df['Water'] / (df['Cement'] + 1e-5)

**Interaction 2: Cement-Age Interaction (Hydration process over time)**

In [5]:
df['Cement_Age_Interaction'] = df['Cement'] * df['Age']

**Transformation: Log-transform for 'Age' (To linearize the logarithmic growth)**

In [6]:
df['Log_Age'] = np.log1p(df['Age'])

print("\nNew Features Created:")
print(df[['Water_Cement_Ratio', 'Cement_Age_Interaction', 'Log_Age']].head())


New Features Created:
   Water_Cement_Ratio  Cement_Age_Interaction   Log_Age
0            0.300000                 15120.0  3.367296
1            0.300000                 15120.0  3.367296
2            0.685714                 89775.0  5.602119
3            0.685714                121362.5  5.902633
4            0.966767                 71496.0  5.888878


Adding 'Water_Cement_Ratio' allows the model to capture the chemical balance of the mixture. 'Log_Age' helps linear models understand the diminishing returns of strength gain over long periods.

#**DATASET SPLITTING (TRAIN/TEST)**

Separate the target variable and split into training/testing sets.

In [7]:
X = df.drop('Strength', axis=1)
y = df['Strength']

80% Training, 20% Testing with a fixed random_state for reproducibility

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTraining set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")


Training set size: 804 samples
Testing set size: 201 samples


Splitting before scaling is crucial to prevent "Data Leakage" (where information from the test set leaks into the training process through the mean/standard deviation).

#**ROBUST FEATURE SCALING**

Normalize numerical ranges using StandardScaler.

In [9]:
scaler = StandardScaler()

Fit only on the training data, then transform both

In [10]:
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

Caling is mandatory for SVR (Support Vector Regression) and Ridge/Lasso. Without scaling, features with large values (like CoarseAggregate ~1000) would overpower smaller ones (like Water ~150).

#**DATA EXPORT FOR MODELING**

Export 4 files for the final training and evaluation stage.


In [11]:
X_train_scaled.to_csv('X_train.csv', index=False)
X_test_scaled.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print("Exported: X_train.csv, X_test.csv, y_train.csv, y_test.csv")

Exported: X_train.csv, X_test.csv, y_train.csv, y_test.csv
